# PRETRAINING (SELF-SUPERVISED)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np

# -----------------------------
# DATASET (LESS OVERLAP = FASTER)
# -----------------------------
class SeismicDataset3D(Dataset):
    def __init__(self, volume, cube_size=128, stride=128):  # 🔥 stride=128 (no overlap)
        self.volume = volume
        self.cs = cube_size

        D, H, W = volume.shape

        self.indices = [
            (z, y, x)
            for z in range(0, D - cube_size, stride)
            for y in range(0, H - cube_size, stride)
            for x in range(0, W - cube_size, stride)
        ]

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        z, y, x = self.indices[idx]
        cube = self.volume[z:z+128, y:y+128, x:x+128].astype(np.float32)
        cube = np.expand_dims(cube, axis=0)
        return torch.from_numpy(cube)

# -----------------------------
# LIGHTER UNET++ 3D (FASTER)
# -----------------------------
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv3d(in_ch, out_ch, 3, padding=1),
            nn.InstanceNorm3d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv3d(out_ch, out_ch, 3, padding=1),
            nn.InstanceNorm3d(out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)

class UNetPP3D(nn.Module):
    def __init__(self):
        super().__init__()

        self.pool = nn.MaxPool3d(2)

        # 🔥 reduced channels (major speed boost)
        self.conv00 = ConvBlock(1, 8)
        self.conv10 = ConvBlock(8, 16)
        self.conv20 = ConvBlock(16, 32)

        self.up10 = nn.ConvTranspose3d(16, 8, 2, stride=2)
        self.up20 = nn.ConvTranspose3d(32, 16, 2, stride=2)

        self.conv01 = ConvBlock(16, 8)
        self.conv11 = ConvBlock(32, 16)

        self.up11 = nn.ConvTranspose3d(16, 8, 2, stride=2)
        self.conv02 = ConvBlock(24, 8)

        self.final = nn.Conv3d(8, 1, 1)

    def forward(self, x):
        x00 = self.conv00(x)
        x10 = self.conv10(self.pool(x00))
        x20 = self.conv20(self.pool(x10))

        x01 = self.conv01(torch.cat([x00, self.up10(x10)], dim=1))
        x11 = self.conv11(torch.cat([x10, self.up20(x20)], dim=1))

        x02 = self.conv02(torch.cat([x00, x01, self.up11(x11)], dim=1))

        return torch.sigmoid(self.final(x02))

# -----------------------------
# DISCONTINUITY TARGET
# -----------------------------
def discontinuity_map(x):
    d1 = x[:,:,1:,:,:] - x[:,:,:-1,:,:]
    d2 = d1[:,:,1:,:,:] - d1[:,:,:-1,:,:]
    d2 = F.pad(d2, (0,0,0,0,0,2))
    return d2.abs()

# -----------------------------
# TRAIN (FAST)
# -----------------------------
def pretrain(volume, epochs=30):
    dataset = SeismicDataset3D(volume)

    loader = DataLoader(
        dataset,
        batch_size=2,                # 🔥 increase if memory allows
        shuffle=True,
        num_workers=4,              # 🔥 parallel loading
        pin_memory=True
    )

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = UNetPP3D().to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

    # 🔥 MIXED PRECISION
    scaler = torch.cuda.amp.GradScaler()

    for epoch in range(epochs):
        losses = []

        for cube in loader:
            cube = cube.to(device, non_blocking=True)

            cube = (cube - cube.mean()) / (cube.std() + 1e-6)

            optimizer.zero_grad()

            # 🔥 AMP forward
            with torch.cuda.amp.autocast():
                pred = model(cube)

                target = discontinuity_map(cube)
                target = target / (target.max() + 1e-6)

                loss = F.l1_loss(pred, target)

            if torch.isnan(loss):
                continue

            # 🔥 AMP backward
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 0.5)
            scaler.step(optimizer)
            scaler.update()

            losses.append(loss.item())

        print(f"[FAST Pretrain] Epoch {epoch+1} | Loss: {np.mean(losses):.4f}")

    return model

# -----------------------------
# RUN
# -----------------------------
volume = np.load("../data/kerry3d_volume.npy")
volume = (volume - volume.min())/(volume.max()-volume.min())

print("Starting FAST Pretraining...")
model = pretrain(volume, epochs=0)

torch.save(model.state_dict(), "unetpp3d_fast_pretrained.pth")
print("Done.")

In [ ]:
import torch
import torch.nn.functional as F
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import os
from torch.utils.data import Dataset, DataLoader

# -----------------------------
# PATHS
# -----------------------------
train_root = "../data/train-20260410T053600Z-3-001/train"
val_root   = "../data/validation-20260410T053559Z-3-001/validation"

# -----------------------------
# LOAD .DAT FILE
# -----------------------------
def load_dat(path, shape=(128,128,128)):
    data = np.fromfile(path, dtype=np.float32)
    data = data.reshape(shape)
    return data

# -----------------------------
# DATASET
# -----------------------------
class FaultDataset(Dataset):
    def __init__(self, root):
        self.seis_dir = os.path.join(root, "seis")
        self.fault_dir = os.path.join(root, "fault")

        self.files = sorted(os.listdir(self.seis_dir))

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        fname = self.files[idx]

        seis_path = os.path.join(self.seis_dir, fname)
        fault_path = os.path.join(self.fault_dir, fname)

        seis = load_dat(seis_path)
        fault = load_dat(fault_path)

        # normalize seismic
        seis = (seis - seis.mean()) / (seis.std() + 1e-6)

        # ensure binary mask
        fault = (fault > 0).astype(np.float32)

        seis = np.expand_dims(seis, 0)
        fault = np.expand_dims(fault, 0)

        return torch.tensor(seis), torch.tensor(fault)

# -----------------------------
# LOAD DATA
# -----------------------------
train_dataset = FaultDataset(train_root)
val_dataset   = FaultDataset(val_root)

train_loader = DataLoader(train_dataset, batch_size=1, shuffle=True, num_workers=4, pin_memory=True)
val_loader   = DataLoader(val_dataset, batch_size=1, shuffle=False, num_workers=2)

# -----------------------------
# MODEL
# -----------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = UNetPP3D().to(device)
model.load_state_dict(torch.load("unetpp3d_fast_pretrained.pth"))
model.train()

optimizer = torch.optim.Adam(model.parameters(), lr=2e-5)

# -----------------------------
# BALANCED BCE
# -----------------------------
def balanced_bce(pred, target):
    pred = torch.clamp(pred, 1e-5, 1-1e-5)

    pos = target.sum()
    neg = target.numel() - pos

    weight_pos = neg / (pos + 1e-6)
    weight_neg = pos / (neg + 1e-6)

    loss = -(
        weight_pos * target * torch.log(pred) +
        weight_neg * (1 - target) * torch.log(1 - pred)
    ).mean()

    return loss

# -----------------------------
# METRIC (IOU)
# -----------------------------
def compute_iou(pred, target, thresh=0.5):
    pred = (pred > thresh).float()
    inter = (pred * target).sum()
    union = pred.sum() + target.sum() - inter
    return (inter / (union + 1e-6)).item()

# -----------------------------
# VISUALIZATION
# -----------------------------
def visualize(x, pred, gt):
    x = x.cpu().numpy()
    pred = pred.detach().cpu().numpy()
    gt = gt.cpu().numpy()

    k = x.shape[2] // 2

    plt.figure(figsize=(15,5))
    plt.subplot(1,3,1); plt.imshow(x[0,0,k], cmap='bone'); plt.title("Input")
    plt.subplot(1,3,2); plt.imshow(pred[0,0,k], cmap='hot'); plt.title("Prediction")
    plt.subplot(1,3,3); plt.imshow(gt[0,0,k], cmap='gray'); plt.title("GT")
    plt.show()

# -----------------------------
# TRAIN + VALIDATION
# -----------------------------
def finetune(epochs=30):
    for epoch in range(epochs):

        # -------- TRAIN --------
        model.train()
        train_losses = []

        for i, (cube, label) in enumerate(train_loader):
            cube = cube.to(device, non_blocking=True)
            label = label.to(device, non_blocking=True)

            optimizer.zero_grad()

            pred = model(cube)

            loss = balanced_bce(pred, label)

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 0.5)
            optimizer.step()

            train_losses.append(loss.item())

            # visualize only first batch
            if i == 0:
                visualize(cube, pred, label)

        # -------- VALIDATION --------
        model.eval()
        val_losses = []
        ious = []

        with torch.no_grad():
            for cube, label in val_loader:
                cube = cube.to(device)
                label = label.to(device)

                pred = model(cube)

                loss = balanced_bce(pred, label)
                iou = compute_iou(pred, label)

                val_losses.append(loss.item())
                ious.append(iou)

        print(f"[Epoch {epoch+1}] "
              f"Train Loss: {np.mean(train_losses):.4f} | "
              f"Val Loss: {np.mean(val_losses):.4f} | "
              f"IoU: {np.mean(ious):.4f}")

# -----------------------------
# RUN
# -----------------------------
print("Start Fine-tuning...")
finetune(epochs=30)

torch.save(model.state_dict(), "fault_finetuned.pth")
print("Done.")

In [ ]:
import torch
import numpy as np
import torch.nn.functional as F

# -----------------------------
# LOAD MODEL
# -----------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = UNetPP3D().to(device)
model.load_state_dict(torch.load("fault_finetuned.pth"))
model.eval()

# -----------------------------
# LOAD VOLUME
# -----------------------------
volume = np.load("../data/kerry3d_volume.npy")
volume = (volume - volume.min())/(volume.max() - volume.min())

D, H, W = volume.shape

# -----------------------------
# PARAMETERS
# -----------------------------
cube_size = 128
stride = 64   # overlap for smooth stitching

# -----------------------------
# OUTPUT ARRAYS
# -----------------------------
pred_volume = np.zeros((D, H, W), dtype=np.float32)
count_map   = np.zeros((D, H, W), dtype=np.float32)

# -----------------------------
# SLIDING WINDOW INFERENCE
# -----------------------------
with torch.no_grad():
    for z in range(0, D - cube_size, stride):
        for y in range(0, H - cube_size, stride):
            for x in range(0, W - cube_size, stride):

                cube = volume[z:z+cube_size, y:y+cube_size, x:x+cube_size]
                cube = (cube - cube.mean()) / (cube.std() + 1e-6)

                cube = torch.tensor(cube).unsqueeze(0).unsqueeze(0).to(device)

                pred = model(cube)
                pred = pred.squeeze().cpu().numpy()

                pred_volume[z:z+cube_size, y:y+cube_size, x:x+cube_size] += pred
                count_map[z:z+cube_size, y:y+cube_size, x:x+cube_size] += 1

# -----------------------------
# NORMALIZE OVERLAPS
# -----------------------------
pred_volume = pred_volume / (count_map + 1e-6)

# -----------------------------
# 🔥 BINARY MASK
# -----------------------------
threshold = 0.5   # you can tune this
binary_mask = (pred_volume > threshold).astype(np.uint8)

# -----------------------------
# SAVE RESULTS
# -----------------------------
np.save("fault_probability.npy", pred_volume)
np.save("fault_binary_mask.npy", binary_mask)

print("Prediction Done.")
print("Saved: fault_probability.npy and fault_binary_mask.npy")

In [ ]:
import matplotlib.pyplot as plt

k = pred_volume.shape[0] // 2

plt.figure(figsize=(15,5))

plt.subplot(1,3,1)
plt.imshow(volume[k], cmap='bone')
plt.title("Seismic")

plt.subplot(1,3,2)
plt.imshow(pred_volume[k], cmap='hot')
plt.title("Probability")

plt.subplot(1,3,3)
plt.imshow(binary_mask[k], cmap='gray')
plt.title("Binary Fault Mask")

plt.show()